# 09. Final Demo

## Goal

Connect the project pieces into one end-to-end meeting assistant demo.

## Why This Step Matters

The final demo shows the complete flow: transcript rows are cleaned, split into chunks, searched by question, and turned into a meeting-minute style answer.

## Inputs

- A meeting audio file for the CLI path, or sample transcript rows for this notebook check.
- A user question.
- `configs/final_demo.yaml`.

## Outputs

- Cleaned transcript rows.
- Retrieved transcript context.
- Final meeting-minute style answer.

## Setup

The notebook uses built-in sample rows so it can verify the final glue without downloading or running an ASR model.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

from src.pipeline.run_demo import DEMO_ROWS, load_config, run_demo_from_rows

CONFIG_PATH = PROJECT_ROOT / "configs" / "final_demo.yaml"
QUESTION = "ที่ประชุมมีมติเรื่องงบประมาณอย่างไร"

config = load_config(CONFIG_PATH)
print("Project root:", PROJECT_ROOT)
print("Config:", config)

Project root: /home/lataeq/Code/Saranote
Config: {'asr_backend': 'typhoon_api', 'model_name': 'openai/whisper-medium', 'language': 'thai', 'task': 'transcribe', 'typhoon_model': 'typhoon-asr-realtime', 'typhoon_api_base_url': 'https://api.opentyphoon.ai/v1', 'typhoon_api_key_env': 'TYPHOON_API_KEY', 'sample_rate': 16000, 'chunk_seconds': 8, 'overlap_seconds': 1, 'max_chunk_characters': 70, 'top_k': 1, 'max_summary_points': 1, 'silence_threshold': 0.001}


## Load Data

Use the same sample transcript rows as the CLI `--demo` mode.

In [2]:
pd.DataFrame(DEMO_ROWS)

,timestamp,text
0,00:00:00 - 00:00:08,เปิดประชุม และแจ้งวาระวันนี้
1,00:00:08 - 00:00:18,ที่ประชุม มีมติ ให้ปรับแผน งบประมาณ สำหรับโครง...
2,00:00:18 - 00:00:28,ฝ่ายดำเนินงานต้องส่งเอกสารภายในวันศุกร์


## Run Experiment

Run the final demo pipeline on the sample transcript rows.

In [3]:
result = run_demo_from_rows(DEMO_ROWS, QUESTION, config)
print("Question:", result["question"])
print("Answer:", result["answer"])

Question: ที่ประชุมมีมติเรื่องงบประมาณอย่างไร
Answer: จาก transcript ที่เกี่ยวข้อง: ที่ประชุม มีมติ ให้ปรับแผน งบประมาณ สำหรับโครงการใหม่


## Inspect Results

Check what transcript context was retrieved before the answer was produced.

In [4]:
pd.DataFrame(result["retrieved_context"])

,score,matched_terms,start_timestamp,end_timestamp,text
0,0.53125,(),00:00:08 - 00:00:18,00:00:08 - 00:00:18,ที่ประชุม มีมติ ให้ปรับแผน งบประมาณ สำหรับโครง...


## Demo Checks

The final answer should use the budget decision and avoid inventing details.

In [5]:
demo_checks = [
    {"check": "has answer", "passed": bool(result["answer"])},
    {"check": "mentions budget", "passed": "งบประมาณ" in result["answer"]},
    {"check": "mentions decision", "passed": "มติ" in result["answer"] or "ปรับแผน" in result["answer"]},
    {"check": "does not invent amount", "passed": "บาท" not in result["answer"] and "ล้าน" not in result["answer"]},
]

demo_checks_df = pd.DataFrame(demo_checks)
print(f"Passed {demo_checks_df['passed'].sum()}/{len(demo_checks_df)} final demo checks")
demo_checks_df

Passed 4/4 final demo checks


,check,passed
0,has answer,True
1,mentions budget,True
2,mentions decision,True
3,does not invent amount,True


In [6]:
assert demo_checks_df["passed"].all(), "Some final demo checks failed"

summary = pd.DataFrame([
    {
        "cleaned_rows": len(result["cleaned_rows"]),
        "retrieved_chunks": len(result["retrieved_context"]),
        "answer": result["answer"],
    }
])
summary

,cleaned_rows,retrieved_chunks,answer
0,3,1,จาก transcript ที่เกี่ยวข้อง: ที่ประชุม มีมติ ...


## Run With Audio

```bash
export TYPHOON_API_KEY="..."

PYTHONPATH=. .venv/bin/python -m src.pipeline.run_demo \
  --audio path/to/audio.wav \
  --question "ที่ประชุมมีมติเรื่องใดบ้าง" \
  --config configs/final_demo.yaml
```

## What We Learned

- The demo pipeline can connect ASR output, cleaning, retrieval, and summarization.
- The notebook verifies the non-ASR glue cheaply.
- The CLI path is ready for real audio when an audio file is available.

## Common Problems

- ASR can be slow because it loads Whisper.
- Poor ASR text will limit retrieval and summary quality.
- Generated outputs under `outputs/` should stay out of Git.

## Next Step

Use Phase 11 error analysis to separate ASR errors, retrieval misses, and summarization failures.